# Notebook 06 — SkinX: lightweight, offline model and inference preparation

**Goal:** Keep the locked Notebook 04 EfficientNetB0 and its validation-selected decision threshold unchanged; convert it to a lightweight Android TFLite model that can screen entirely offline. The finished app will not need internet, FastAPI, or a cloud account to screen a lesion.

**Steps:** Verify the locked files → preserve Notebook 04's exact RGB/224×224 resize-and-pad pipeline → try TFLite formats → compare each on the **original conventional validation split** → freeze the smallest candidate with no loss of validation TP or TN → create and test reusable inference/decision/input-safety code → export a mobile-only package and **optional** Python/FastAPI reference → descriptive PWA check and full reproducibility records.

**Clinical scope:** This is a **binary research screening model**, not an SCC/BCC/AK diagnosis. `TARGET` means a suspicious lesion pattern; `NON_TARGET` is not proof of no cancer. **Cannot Infer** is used for technical failures only, until separate uncertainty / image-quality rejection criteria are legitimately validated from appropriate development data. Never fit thresholds to PWA evaluation images or the sealed conventional test split.

**Low-cost device requirement:** Small storage footprint and offline operation are necessary, but Colab CPU speed is **not** proof of old Android CPU speed, RAM use, or battery life. Native Flutter/LiteRT parity and real-device benchmarks still have to be done before claiming mobile readiness.

## 1. Connect Drive and set the run

Run top to bottom in Colab. **CPU mode is sufficient** for conversion and evaluation. Everything runs against your existing Drive layout; **Notebook 04 model and data remain unchanged**. Change `RUN_TAG` only when intentionally starting a fresh experiment.

In [1]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score
from IPython.display import display

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/SkinX')
SPLITS = ROOT / 'data/splits'
LOCKED_RUN = ROOT / 'models/runs/notebook_04/three_models_finetune_v1'
PWA_ROOT = ROOT / 'data/raw_data/PWA_clinical_images'
RUN_TAG = 'offline_tflite_v2'        # Change this only if starting a new run.
OUT = ROOT / 'models/runs/notebook_06' / RUN_TAG

TRAIN_CSV = SPLITS / 'skinX_train_manifest_v1.csv'
VAL_CSV = SPLITS / 'skinX_val_manifest_v1.csv'
LOCK_JSON = LOCKED_RUN / 'locked_selection.json'
IMAGE_SIZE = (224, 224)
SEED = 42
CALIBRATION_PER_CLASS = 75        # Training images ONLY, never validation/test/PWA.
NUM_THREADS = 2                   # Basic CPU setup; device test comes later.

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Destination:', OUT)


Mounted at /content/drive
TensorFlow: 2.21.0
Destination: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2


## 2. Verify Notebook 04's locked model and data
We do **not** read the sealed conventional test split. We use the **training split only** for INT8 calibration and the original **validation split only** for choosing a deployment format. The PWA dataset is considered only after that choice is frozen.

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

for path in [TRAIN_CSV, VAL_CSV, LOCK_JSON]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required file: {path}')
lock = json.loads(LOCK_JSON.read_text(encoding='utf-8'))
MODEL_PATH = LOCKED_RUN / lock['model_file']
if not MODEL_PATH.is_file() or sha256(MODEL_PATH) != lock['model_sha256']:
    raise ValueError('Locked model missing or changed; do not continue.')
if lock['architecture'] != 'EfficientNetB0':
    raise ValueError('Expected locked EfficientNetB0 from Notebook 04')
if sha256(TRAIN_CSV) != lock['train_sha256'] or sha256(VAL_CSV) != lock['validation_sha256']:
    raise ValueError('Training/validation CSV changed since Notebook 04')
THRESHOLD = float(lock['threshold'])
assert 0 < THRESHOLD < 1

train = pd.read_csv(TRAIN_CSV)
val = pd.read_csv(VAL_CSV)
required = ['image_id', 'filepath', 'binary_label', 'label_group', 'group_id', 'md5_hash']
for name, frame in [('train', train), ('validation', val)]:
    if not set(required).issubset(frame.columns):
        raise ValueError(f'{name}: required CSV columns are missing')
    if frame[required].isna().any().any():
        raise ValueError(f'{name}: missing required fields')
    if not frame['binary_label'].isin([0, 1]).all():
        raise ValueError(f'{name}: incorrect binary labels')
    if not frame['binary_label'].eq(frame['label_group'].map({'NON_TARGET':0, 'TARGET':1})).all():
        raise ValueError(f'{name}: label mismatch')
    if frame['md5_hash'].duplicated().any() or frame['image_id'].duplicated().any():
        raise ValueError(f'{name}: duplicate IDs or hashes')
    absent = [p for p in frame['filepath'] if not Path(p).is_file()]
    if absent:
        raise FileNotFoundError(f'{name}: {len(absent)} missing files, example: {absent[0]}')
for col in ['image_id', 'md5_hash', 'group_id']:
    if set(train[col]) & set(val[col]):
        raise ValueError(f'Data leak: train and validation overlap in {col}')

# Refuse to overwrite an earlier deployment experiment.
if OUT.exists() and any(OUT.iterdir()):
    raise FileExistsError(f'{OUT} already contains files. Set a NEW RUN_TAG.')
OUT.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
if tuple(model.input_shape[1:]) != (224,224,3) or model.output_shape[-1] != 1:
    raise ValueError(f'Unexpected locked input/output: {model.input_shape} -> {model.output_shape}')
print('Original model:', lock['stage'])
print('Threshold:', THRESHOLD)
print('Training:', len(train), 'Validation:', len(val))
print('Output shape:', model.output_shape)


Original model: EfficientNetB0__head
Threshold: 0.374953031539917
Training: 1661 Validation: 339
Output shape: (None, 1)


## 3. Preserve training preprocessing

Training used `tf.io.decode_image(..., channels=3)` → float32 RGB values in **0–255** → `tf.image.resize_with_pad` (bilinear, centered black padding) to **224 × 224**. EfficientNetB0 preprocessing is already inside the locked network: **do not divide by 255**. The Android app must recreate the same tensor, including EXIF orientation handling of phone photos **before** the resize. Native parity is not assumed; we export a synthetic golden fixture to test it.

In [3]:
def load_image(path):
    data = tf.io.read_file(str(path))
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), *IMAGE_SIZE)
    return image.numpy()

def get_images(frame):
    # Loading one image at a time keeps memory usage small on Colab.
    for p in frame['filepath'].astype(str):
        yield load_image(p)

def metrics(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores >= THRESHOLD).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel())
    return {'n': int(len(y_true)), 'tp': tp, 'fn': fn, 'tn': tn, 'fp': fp,
            'sensitivity': tp/(tp+fn) if tp+fn else None,
            'specificity': tn/(tn+fp) if tn+fp else None,
            'auroc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true))==2 else None}

def keras_scores(frame):
    results=[]
    for img in get_images(frame):
        score=float(np.asarray(model(np.expand_dims(img,0), training=False)).reshape(-1)[0])
        results.append(score)
    return np.asarray(results, dtype=np.float32)

truth = val['binary_label'].astype(int).to_numpy()
keras_val = keras_scores(val)
reference = metrics(truth, keras_val)
expected = lock['validation_metrics']
if (reference['tp'], reference['tn']) != (expected['true_positives'], expected['true_negatives']):
    raise RuntimeError('Loaded Keras model does not reproduce Notebook 04 validation labels at locked threshold. Check environment before converting.')
print('Verified original Keras validation:')
display(pd.DataFrame([reference]))

# Training examples only; class-balanced calibration, no leaked PWA or test images.
calibration = (train.groupby('binary_label', group_keys=False)
               .sample(n=CALIBRATION_PER_CLASS, random_state=SEED)
               .sample(frac=1, random_state=SEED))
if len(calibration) != 2*CALIBRATION_PER_CLASS:
    raise ValueError('Not enough calibration images per class')

def representative_dataset():
    for image in get_images(calibration):
        yield [np.expand_dims(image, 0).astype(np.float32)]

print('INT8 calibration:', len(calibration), 'TRAIN images only')


Verified original Keras validation:


,n,tp,fn,tn,fp,sensitivity,specificity,auroc
0,339,269,17,43,10,0.940559,0.811321,0.947552


INT8 calibration: 150 TRAIN images only


## 4. Convert into several offline candidates
Four options: full float32, float16 weights, dynamic-range INT8 weights, and full INT8. Full INT8 uses integer input/output and train-only calibration. **Conversion failures are recorded**, not hidden by enabling `SELECT_TF_OPS` / Flex (which would complicate the lightweight Flutter app).

The resulting sizes and speeds are **measured**, not assumed. Float16 can be smaller without being faster on an old CPU. INT8 may be faster but can change a borderline prediction. Compare before choosing.

In [4]:
variants = ['float32', 'float16', 'dynamic_int8', 'full_int8']
models_dir = OUT / 'candidates'
models_dir.mkdir(exist_ok=True)
conversion = {}

for variant in variants:
    print('Converting:', variant)
    started = time.perf_counter()
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
        if variant == 'float16':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.target_spec.supported_types = [tf.float16]
        elif variant == 'dynamic_int8':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
        elif variant == 'full_int8':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.uint8
            converter.inference_output_type = tf.uint8
        raw = converter.convert()
        path = models_dir / f'skinx_{variant}.tflite'
        path.write_bytes(raw)
        conversion[variant] = {'status': 'converted', 'path': str(path),
                               'size_mb': round(path.stat().st_size/1024**2, 3),
                               'seconds': round(time.perf_counter()-started, 1)}
        print('  OK:', conversion[variant]['size_mb'], 'MiB')
    except Exception as e:
        conversion[variant] = {'status': 'FAILED', 'error': str(e)[:1200]}
        print('  Failed:', type(e).__name__, str(e)[:250])
    gc.collect()

(OUT/'conversion_report.json').write_text(json.dumps(conversion, indent=2)+'\n')
display(pd.DataFrame.from_dict(conversion, orient='index'))
if not any(v['status']=='converted' for v in conversion.values()):
    raise RuntimeError('No TFLite model could be converted. Keep the Keras model unchanged.')


Converting: float32
Saved artifact at '/tmp/tmpji81pihy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135658990798032: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135658990797840: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135658997677904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658980999632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658981000208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658980999824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658980999440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658981000592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658981000976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135658981000784: TensorSpec(shape=(), dtype=tf.resour

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  OK: 4.705 MiB


,status,path,size_mb,seconds
float32,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,15.320,15.3
float16,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,7.754,15.5
dynamic_int8,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,4.360,15.7
full_int8,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,4.705,205.1


## 5. Run the TFLite interpreter and verify the input types
The app must detect the input tensor type: float models use `float32` RGB 0–255; the integer model needs `input / scale + zero_point` and corresponding output dequantization. **Never guess integer tensor scales or normalize to 0–1.**

In [5]:
def interpreter_for(path):
    interpreter = tf.lite.Interpreter(model_path=str(path), num_threads=NUM_THREADS)
    interpreter.allocate_tensors()
    ins, outs = interpreter.get_input_details(), interpreter.get_output_details()
    if len(ins)!=1 or len(outs)!=1 or tuple(ins[0]['shape'])!=(1,224,224,3):
        raise ValueError('Unexpected TFLite input/output tensor contract')
    if int(np.prod(outs[0]['shape'])) != 1:
        raise ValueError('Expected one output probability')
    op_names = [op['op_name'] for op in interpreter._get_ops_details()]
    if any(op.startswith('Flex') or 'CUSTOM' in op for op in op_names):
        raise ValueError('TFLite uses Flex/custom ops; not suitable for default lightweight runtime')
    return interpreter, ins[0], outs[0]

def run_one(interpreter, inp, out, image):
    data = np.expand_dims(image,0).astype(np.float32)
    if np.issubdtype(inp['dtype'], np.integer):
        scale, zero = inp['quantization']
        if scale<=0: raise ValueError('Invalid quantized input scale')
        bounds = np.iinfo(inp['dtype'])
        data = np.clip(np.rint(data/scale+zero), bounds.min, bounds.max).astype(inp['dtype'])
    else:
        data = data.astype(inp['dtype'])
    interpreter.set_tensor(inp['index'], data)
    interpreter.invoke()
    p = interpreter.get_tensor(out['index'])
    if np.issubdtype(out['dtype'], np.integer):
        scale, zero = out['quantization']
        if scale<=0: raise ValueError('Invalid quantized output scale')
        p = (p.astype(np.float32)-zero)*scale
    score = float(np.asarray(p).reshape(-1)[0])
    if not np.isfinite(score) or score < -0.005 or score > 1.005:
        raise ValueError('Invalid output probability')
    return min(1.0, max(0.0, score))

# Brief smoke test before evaluating full validation data.
example_image = load_image(val.iloc[0]['filepath'])
ready = {}
for variant, info in conversion.items():
    if info['status'] != 'converted': continue
    try:
        intr, inp, out = interpreter_for(info['path'])
        score = run_one(intr, inp, out, example_image)
        ready[variant] = True
        info['input_dtype'] = str(inp['dtype'])
        info['output_dtype'] = str(out['dtype'])
        info['input_quantization'] = list(map(float, inp['quantization']))
        info['output_quantization'] = list(map(float, out['quantization']))
        print(f'{variant}: first score {score:.5f} | input {inp["dtype"]} | output {out["dtype"]}')
    except Exception as e:
        ready[variant] = False
        info['status'] = 'FAILED_SMOKE_TEST'
        info['error'] = str(e)[:1200]
        print(f'{variant}: not usable — {e}')


float32: first score 0.86364 | input <class 'numpy.float32'> | output <class 'numpy.float32'>


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


float16: first score 0.86393 | input <class 'numpy.float32'> | output <class 'numpy.float32'>
dynamic_int8: first score 0.90437 | input <class 'numpy.float32'> | output <class 'numpy.float32'>
full_int8: first score 0.91406 | input <class 'numpy.uint8'> | output <class 'numpy.uint8'>


## 6. Compare performance at the **same original threshold**
Run every usable variant on the **same conventional validation images**. Do not adjust the threshold to improve a quantized model. The acceptance rule is strict: **no fewer true positives and no fewer true negatives than the verified Keras model**. This is a technical conversion check, **not** a clinical guarantee.

In [6]:
report=[]
val_predictions=val[['image_id','filepath','binary_label','diagnostic']].copy()
val_predictions['keras_probability'] = keras_val

for variant, good in ready.items():
    if not good: continue
    info=conversion[variant]
    intr, inp, out=interpreter_for(info['path'])
    # First load + invoke has setup overhead. Short CPU timing is only a Colab proxy.
    for _ in range(3): run_one(intr, inp, out, example_image)
    times=[]
    for _ in range(20):
        t0=time.perf_counter()
        run_one(intr, inp, out, example_image)
        times.append((time.perf_counter()-t0)*1000)
    scores=np.asarray([run_one(intr,inp,out,img) for img in get_images(val)], dtype=np.float32)
    stats=metrics(truth, scores)
    accepted=stats['tp']>=reference['tp'] and stats['tn']>=reference['tn']
    info['accepted_on_validation']=bool(accepted)
    info['median_colab_inference_ms']=round(float(np.median(times)),2)
    info['max_absolute_probability_difference']=float(np.max(np.abs(scores-keras_val)))
    val_predictions[f'{variant}_probability']=scores
    report.append({'variant':variant,'size_mb':info['size_mb'],
                   'median_colab_inference_ms':info['median_colab_inference_ms'],
                   'max_probability_difference':info['max_absolute_probability_difference'],
                   'accepted':accepted,**stats})

compare=pd.DataFrame(report).sort_values('size_mb') if report else pd.DataFrame()
compare.to_csv(OUT/'validation_comparison.csv', index=False)
val_predictions.to_csv(OUT/'validation_probabilities.csv', index=False)
(OUT/'conversion_report.json').write_text(json.dumps(conversion,indent=2)+'\n')
print('Reference Keras confusion:', reference)
display(compare)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

Reference Keras confusion: {'n': 339, 'tp': 269, 'fn': 17, 'tn': 43, 'fp': 10, 'sensitivity': 0.9405594405594405, 'specificity': 0.8113207547169812, 'auroc': 0.9475524475524476}


,variant,size_mb,median_colab_inference_ms,max_probability_difference,accepted,n,tp,fn,tn,fp,sensitivity,specificity,auroc
2,dynamic_int8,4.360,16.55,0.119101,False,339,268,18,41,12,0.937063,0.773585,0.946497
3,full_int8,4.705,7.22,0.527795,False,339,267,19,33,20,0.933566,0.622642,0.922054
1,float16,7.754,10.39,0.004999,True,339,269,17,43,10,0.940559,0.811321,0.947289
0,float32,15.320,10.66,0.002024,False,339,268,18,43,10,0.937063,0.811321,0.947486


## 7. Freeze ONE deployment model
**Policy:** Choose the **smallest accepted** TFLite candidate, breaking size ties by lower Colab latency. If all conversions reduce validation performance, **stop** instead of quietly accepting a loss. You may investigate conversion compatibility in a *new* experiment, still using training/validation only. **Never tune on the held-out PWA or conventional test images.**

In [7]:
accepted = compare[compare['accepted']].copy() if len(compare) else pd.DataFrame()
if accepted.empty:
    print('STOP: No TFLite variant preserved both TP and TN on conventional validation.')
    print('Do not ship one of these models. Review conversion/compatibility in a new run.')
    DEPLOY_READY = False
else:
    chosen=accepted.sort_values(['size_mb','median_colab_inference_ms']).iloc[0]
    variant=str(chosen['variant'])
    source_path=Path(conversion[variant]['path'])
    deploy_dir=OUT/'deployment'
    deploy_dir.mkdir(exist_ok=False)
    dest=deploy_dir/'skinx_screening.tflite'
    shutil.copy2(source_path,dest)
    intr,inp,out=interpreter_for(dest)
    config={
        'schema_version': 1,
        'model_name': 'SkinX offline binary screening',
        'selected_variant': variant,
        'source_architecture': lock['architecture'],
        'source_stage': lock['stage'],
        'source_model_sha256': lock['model_sha256'],
        'tflite_sha256': sha256(dest),
        'model_size_mib': round(dest.stat().st_size/(1024**2), 3),
        'input_shape': [1,224,224,3],
        'input_color_order': 'RGB',
        'external_normalization': 'NONE; pixel values are float32 0..255 before tensor quantization',
        'resize': 'tf.image.resize_with_pad, bilinear, preserve aspect ratio, black padding, centered',
        'input_dtype': str(np.dtype(inp['dtype'])),
        'output_dtype': str(np.dtype(out['dtype'])),
        'input_quantization': {'scale':float(inp['quantization'][0]), 'zero_point':int(inp['quantization'][1])},
        'output_quantization': {'scale':float(out['quantization'][0]), 'zero_point':int(out['quantization'][1])},
        'positive_label': 1,
        'labels': {'0':'NON_TARGET','1':'TARGET'},
        'target_probability_threshold': THRESHOLD,
        'decision_rule': 'TARGET when probability >= target_probability_threshold; otherwise NON_TARGET',
        'cannot_infer': 'Only technical failures e.g. file cannot be decoded; no validated uncertainty or quality cutoff exists',
        'validation_metrics': {k: (bool(v) if isinstance(v,np.bool_) else float(v) if isinstance(v,np.floating) else int(v) if isinstance(v,np.integer) else v) for k,v in chosen.to_dict().items()},
        'tensorflow_version':tf.__version__,
        'limitations':'Research pilot; not diagnostic; PWA data sparse and source-reported; do not hide clinical warning for non-target predictions.',
        'connectivity':'Screening inference is entirely offline when this model and preprocessing are bundled in the Android app.'
    }
    (deploy_dir/'skinx_model_config.json').write_text(json.dumps(config,indent=2)+'\n')
    (deploy_dir/'model_sha256.txt').write_text(config['tflite_sha256']+'  skinx_screening.tflite\n')
    DEPLOY_READY=True
    print('Selected:',variant,'|',config['model_size_mib'],'MiB')
    print('Deployment files:',deploy_dir)


Selected: float16 | 7.754 MiB
Deployment files: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/deployment


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 8. Golden image for Flutter preprocessing and TFLite parity

Use a **synthetic color fixture**, not a patient image, to check preprocessing and one TFLite probability across Android/Python. The fixture PNG is only for engineering tests; it should not be shipped with the released app. The expected float32 resized tensor and output score are saved separately from mobile model assets.

In [8]:
if DEPLOY_READY:
    from PIL import Image
    parity_dir = OUT / 'parity_tests'
    parity_dir.mkdir(exist_ok=True)

    # Synthetic engineering fixture; no patient/lesion image is packaged.
    hh, ww = 151, 271
    yy, xx = np.indices((hh, ww))
    sample = np.stack([
        (xx * 255 // max(ww-1, 1)).astype(np.uint8),
        (yy * 255 // max(hh-1, 1)).astype(np.uint8),
        ((xx+2*yy) % 256).astype(np.uint8)], axis=-1)
    sample[25:67, 80:128, :] = [18, 160, 220]
    synthetic_file = parity_dir / 'synthetic_source.png'
    Image.fromarray(sample, 'RGB').save(synthetic_file)

    golden_input = load_image(synthetic_file)
    np.save(parity_dir/'expected_input_rgb_float32.npy', golden_input)
    intr, inp, out = interpreter_for(deploy_dir/'skinx_screening.tflite')
    golden_score = run_one(intr, inp, out, golden_input)
    golden = {
        'source_fixture': synthetic_file.name,
        'expected_tensor_file': 'expected_input_rgb_float32.npy',
        'expected_shape': list(golden_input.shape),
        'expected_dtype': str(golden_input.dtype),
        'input_min': float(golden_input.min()),
        'input_max': float(golden_input.max()),
        'expected_target_probability': golden_score,
        'expected_decision': 'Suspicious' if golden_score >= THRESHOLD else 'Less suspicious',
        'test_input_sha256': sha256(synthetic_file),
        'mean_absolute_float_difference_tolerance_for_native_resize': 1.0,
        'note': 'Synthetic only. Android must also compare output score and decision at locked threshold. Resize tolerance is an engineering guide, not a clinical equivalence proof.'
    }
    (parity_dir/'expected_output.json').write_text(json.dumps(golden, indent=2)+'\n', encoding='utf-8')
    print('Saved synthetic Android parity fixture:', parity_dir)
else:
    print('Parity fixture skipped: no deployment-ready model.')

/tmp/ipykernel_723/288410607.py:15: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(sample, 'RGB').save(synthetic_file)


Saved synthetic Android parity fixture: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/parity_tests


## 9. Separate **descriptive** PWA check (after the choice is frozen)
This is *not* used to choose or tune the compressed version. PWA labels are **provisional**, some images are small, and only **5 non-target images** exist. Do not treat good pilot numbers as clinical validation.

In [9]:
PWA_CSV=PWA_ROOT/'metadata.csv'
if DEPLOY_READY and PWA_CSV.exists():
    pwa=pd.read_csv(PWA_CSV)
    for field in ['image_id','filepath','binary_label','split_group_id']:
        if field not in pwa.columns: raise ValueError(f'PWA dataset missing {field}')
    pwa['absolute_path']=pwa['filepath'].map(lambda x: str(PWA_ROOT/str(x)))
    missing=[p for p in pwa['absolute_path'] if not Path(p).is_file()]
    if missing: raise FileNotFoundError(f'{len(missing)} PWA images missing: {missing[0]}')
    if pwa['image_id'].duplicated().any(): raise ValueError('Duplicate PWA ids')
    intr,inp,out=interpreter_for(deploy_dir/'skinx_screening.tflite')
    keras_pwa=[]; tflite_pwa=[]
    for path in pwa['absolute_path']:
        img=load_image(path)
        keras_pwa.append(float(np.asarray(model(img[None,...],training=False)).reshape(-1)[0]))
        tflite_pwa.append(run_one(intr,inp,out,img))
    pwa['keras_probability']=keras_pwa
    pwa['tflite_probability']=tflite_pwa
    pwa['keras_prediction']=(pwa['keras_probability']>=THRESHOLD).astype(int)
    pwa['tflite_prediction']=(pwa['tflite_probability']>=THRESHOLD).astype(int)
    pwa.to_csv(OUT/'pwa_descriptive_predictions.csv',index=False)
    print('PWA check — original Keras:',metrics(pwa['binary_label'],pwa['keras_probability']))
    print('PWA check — chosen TFLite:',metrics(pwa['binary_label'],pwa['tflite_probability']))
    print('Changed PWA decisions:',int((pwa['keras_prediction']!=pwa['tflite_prediction']).sum()))
    print('PWA results are for reporting only, not re-selection.')
else:
    print('PWA descriptive check skipped (no deployment-ready variant or PWA metadata missing).')


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


PWA check — original Keras: {'n': 33, 'tp': 28, 'fn': 0, 'tn': 2, 'fp': 3, 'sensitivity': 1.0, 'specificity': 0.4, 'auroc': 0.9785714285714285}
PWA check — chosen TFLite: {'n': 33, 'tp': 28, 'fn': 0, 'tn': 2, 'fp': 3, 'sensitivity': 1.0, 'specificity': 0.4, 'auroc': 0.9785714285714285}
Changed PWA decisions: 0
PWA results are for reporting only, not re-selection.


## 10. Clinical decisions versus technical failures

The user sees **Suspicious**, **Less suspicious**, or **Cannot infer**. The first two are **binary screening labels at the frozen Notebook 04 threshold**, not diagnoses. `Cannot infer` is **only** for failed input checks, decoding errors, or runtime errors. There is **no validated confidence-based abstention or blur/lighting threshold** yet. Even if a result is *Less suspicious*, a changing, bleeding, painful, or non-healing lesion still needs assessment.

The next cell exports a small **reusable Python reference** (for offline testing and optional FastAPI later), with checks for supported JPEG/PNG images, corrupt/truncated files, unusually large payloads, and pixel limits. These last limits are technical safeguards, **not clinically validated quality checks**. Captured phone photos need EXIF orientation normalized before Flutter creates the inference tensor. In the Python reference, this normalization is applied if needed. No image, location, or result is automatically uploaded.

In [10]:
MODULE_CODE = r'''
"""SkinX offline screening reference. NOT the Android inference runtime.

Bundle only skinx_screening.tflite + skinx_model_config.json in Flutter.
Use this Python file for parity checks or an optional server. Never infer diagnoses.
"""
from __future__ import annotations

import hashlib
import io
import json
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError
import tensorflow as tf


class InvalidImage(ValueError):
    """Input rejected for a technical reason, not a medical judgment."""


def _verify_and_orient(data: bytes, max_bytes: int, max_pixels: int) -> bytes:
    if not isinstance(data, bytes) or len(data) == 0:
        raise InvalidImage('EMPTY_IMAGE')
    if len(data) > max_bytes:
        raise InvalidImage('IMAGE_TOO_LARGE_BYTES')
    try:
        with Image.open(io.BytesIO(data)) as image:
            if image.format not in ('JPEG', 'PNG'):
                raise InvalidImage('UNSUPPORTED_IMAGE_FORMAT')
            if getattr(image, 'n_frames', 1) != 1:
                raise InvalidImage('ANIMATED_IMAGE_UNSUPPORTED')
            w, h = image.size
            if w <= 0 or h <= 0 or w*h > max_pixels:
                raise InvalidImage('IMAGE_DIMENSIONS_UNSUPPORTED')
            orientation = image.getexif().get(274, 1)
            # Force full decode: Image.open alone does not catch truncated images.
            image.load()
            if orientation != 1:
                # Phone/gallery orientation is normalized BEFORE 224x224 resizing.
                oriented = ImageOps.exif_transpose(image).convert('RGB')
                stream = io.BytesIO()
                oriented.save(stream, format='PNG')
                data = stream.getvalue()
    except InvalidImage:
        raise
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as exc:
        raise InvalidImage('INVALID_OR_CORRUPT_IMAGE') from exc
    return data


def preprocess_bytes(data: bytes, config: dict) -> np.ndarray:
    """Exact training resize on already oriented bytes; returns float32 RGB 0..255."""
    raw = _verify_and_orient(
        data,
        max_bytes=int(config['max_upload_bytes']),
        max_pixels=int(config['max_image_pixels']),
    )
    try:
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize_with_pad(tf.cast(image, tf.float32), 224, 224)
        return np.asarray(image.numpy(), dtype=np.float32)
    except Exception as exc:
        raise InvalidImage('UNABLE_TO_PREPROCESS_IMAGE') from exc


class ScreeningEngine:
    """One interpreter per instance. Do not share the same instance concurrently."""
    def __init__(self, model_path, config_path, num_threads=2):
        self.model_path = Path(model_path)
        self.config = json.loads(Path(config_path).read_text(encoding='utf-8'))
        if hashlib.sha256(self.model_path.read_bytes()).hexdigest() != self.config['tflite_sha256']:
            raise ValueError('MODEL_HASH_MISMATCH')
        self.threshold = float(self.config['target_probability_threshold'])
        if not 0 < self.threshold < 1 or self.config['labels'] != {'0':'NON_TARGET','1':'TARGET'}:
            raise ValueError('INVALID_SCREENING_CONFIG')
        self.interpreter = tf.lite.Interpreter(model_path=str(self.model_path), num_threads=num_threads)
        self.interpreter.allocate_tensors()
        inputs, outputs = self.interpreter.get_input_details(), self.interpreter.get_output_details()
        if len(inputs) != 1 or len(outputs) != 1 or tuple(inputs[0]['shape']) != (1,224,224,3):
            raise ValueError('UNEXPECTED_MODEL_INPUT')
        if int(np.prod(outputs[0]['shape'])) != 1:
            raise ValueError('UNEXPECTED_MODEL_OUTPUT')
        self.inp, self.out = inputs[0], outputs[0]

    def predict_tensor(self, image: np.ndarray) -> float:
        if image.shape != (224,224,3) or not np.isfinite(image).all():
            raise InvalidImage('INVALID_TENSOR')
        value = np.expand_dims(image, 0).astype(np.float32)
        if np.issubdtype(self.inp['dtype'], np.integer):
            scale, zero = self.inp['quantization']
            if scale <= 0:
                raise ValueError('INVALID_INPUT_QUANTIZATION')
            limits = np.iinfo(self.inp['dtype'])
            value = np.clip(np.rint(value/scale + zero), limits.min, limits.max).astype(self.inp['dtype'])
        else:
            value = value.astype(self.inp['dtype'])
        self.interpreter.set_tensor(self.inp['index'], value)
        self.interpreter.invoke()
        output = self.interpreter.get_tensor(self.out['index'])
        if np.issubdtype(self.out['dtype'], np.integer):
            scale, zero = self.out['quantization']
            if scale <= 0:
                raise ValueError('INVALID_OUTPUT_QUANTIZATION')
            output = (output.astype(np.float32)-zero)*scale
        score = float(np.asarray(output).reshape(-1)[0])
        if not np.isfinite(score) or score < -0.005 or score > 1.005:
            raise ValueError('INVALID_MODEL_SCORE')
        return float(np.clip(score, 0.0, 1.0))

    def screen_bytes(self, data: bytes) -> dict:
        try:
            image = preprocess_bytes(data, self.config)
            score = self.predict_tensor(image)
        except InvalidImage as exc:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':str(exc), 'message':'Please choose another clear JPEG or PNG image.'}
        except Exception:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':'INFERENCE_ERROR', 'message':'Could not process this image. Please try again.'}
        if score >= self.threshold:
            return {'status':'Suspicious', 'target_probability':score, 'reason':None,
                    'message':'This image shows a suspicious pattern. Please seek clinical assessment.'}
        return {'status':'Less suspicious', 'target_probability':score, 'reason':None,
                'message':'This screen cannot rule out skin cancer. Seek care for changing, painful, bleeding or non-healing lesions.'}

    def screen_file(self, file_path) -> dict:
        try:
            path = Path(file_path)
            if path.stat().st_size > int(self.config['max_upload_bytes']):
                raise InvalidImage('IMAGE_TOO_LARGE_BYTES')
            return self.screen_bytes(path.read_bytes())
        except (OSError, InvalidImage) as exc:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':str(exc) if isinstance(exc, InvalidImage) else 'UNREADABLE_IMAGE',
                    'message':'The image could not be opened.'}

'''

if DEPLOY_READY:
    module_path = OUT / 'skinx_inference.py'
    module_path.write_text(MODULE_CODE, encoding='utf-8')
    compile(module_path.read_text(encoding='utf-8'), str(module_path), 'exec')

    # Technical constraints, not clinically validated quality/uncertainty gates.
    config['max_upload_bytes'] = 12 * 1024 * 1024
    config['max_image_pixels'] = 20_000_000
    config['allowed_image_formats'] = ['JPEG','PNG']
    config['technical_image_checks'] = 'File validity, format, dimensions, payload size; no clinical image-quality gate.'
    config['exif_orientation'] = 'Normalize when needed before RGB resize; verify Flutter parity separately.'
    config['uncertainty_abstention_enabled'] = False
    config['cannot_infer_reasons'] = ['EMPTY_IMAGE','IMAGE_TOO_LARGE_BYTES','IMAGE_DIMENSIONS_UNSUPPORTED',
                                     'UNSUPPORTED_IMAGE_FORMAT','ANIMATED_IMAGE_UNSUPPORTED',
                                     'INVALID_OR_CORRUPT_IMAGE','UNABLE_TO_PREPROCESS_IMAGE',
                                     'INFERENCE_ERROR']
    config['threshold_origin'] = 'Notebook 04 conventional validation; locked, never updated using PWA/test.'
    config['display_messages'] = {
        'Suspicious': 'Suspicious pattern. Seek clinical assessment.',
        'Less suspicious': 'This does not rule out cancer. Seek assessment for persistent or changing lesions.',
        'Cannot infer': 'Could not screen this image. Please retake or choose another supported image.'
    }
    (deploy_dir/'skinx_model_config.json').write_text(json.dumps(config,indent=2)+'\n',encoding='utf-8')
    print('Saved reusable Python inference module:', module_path)
else:
    print('No model passed the validation comparison; skipping deployment module.')

Saved reusable Python inference module: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/skinx_inference.py


## 11. Repeatability, validation of invalid inputs, and inference consistency

Run multiple predictions on identical preprocessed tensors; require stable outputs and confirm the new inference API matches the frozen TFLite model. Smoke-test empty, corrupt, unsupported and oversized inputs. This verifies only the **Colab/Python reference**, not native Android or clinical image quality.

In [11]:
if DEPLOY_READY:
    import importlib.util
    spec = importlib.util.spec_from_file_location('skinx_inference', module_path)
    screening_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screening_module)
    engine = screening_module.ScreeningEngine(
        deploy_dir/'skinx_screening.tflite',
        deploy_dir/'skinx_model_config.json',
        num_threads=NUM_THREADS)

    # Image comes from the synthetic fixture, not a patient or evaluation image.
    synthetic_bytes = synthetic_file.read_bytes()
    image_tensor = screening_module.preprocess_bytes(synthetic_bytes, config)
    np.testing.assert_allclose(image_tensor, golden_input, rtol=0, atol=1e-6)
    repeat = np.asarray([engine.predict_tensor(image_tensor) for _ in range(8)],dtype=np.float64)
    drift = float(np.max(np.abs(repeat-repeat[0])))
    if drift > 1e-6:
        raise RuntimeError(f'Non-repeatable local TFLite prediction: max drift={drift}')
    if abs(float(repeat[0])-golden_score) > 1e-6:
        raise RuntimeError('Exported module did not reproduce original TFLite parity score')

    good = engine.screen_bytes(synthetic_bytes)
    assert good['status'] in {'Suspicious','Less suspicious'}
    bad_cases = {
        'empty':b'',
        'invalid_bytes':b'not an image',
        'unsupported_format':None,
    }
    from PIL import Image
    import io
    test_webp = io.BytesIO()
    try:
        Image.new('RGB', (25,35),(40,140,90)).save(test_webp,format='WEBP')
        bad_cases['unsupported_format']=test_webp.getvalue()
    except Exception:
        bad_cases.pop('unsupported_format')
    for case, raw in bad_cases.items():
        result = engine.screen_bytes(raw)
        if result['status'] != 'Cannot infer' or result['target_probability'] is not None:
            raise AssertionError(f'{case} unexpectedly returned {result}')
    # Technical payload limit (not a medical image quality heuristic).
    oversized = b'1'*(int(config['max_upload_bytes'])+1)
    assert engine.screen_bytes(oversized)['status'] == 'Cannot infer'

    # Check one REAL saved validation image through the reusable bytes API.
    # It must not change the original Keras/TFLite validation selection.
    sample_path = Path(val.iloc[0]['filepath'])
    prediction = engine.screen_file(sample_path)
    assert prediction['status'] in {'Suspicious','Less suspicious'}
    sample_resized = screening_module.preprocess_bytes(sample_path.read_bytes(),config)
    direct_score = run_one(*interpreter_for(dest), sample_resized)
    if abs(prediction['target_probability'] - direct_score) > 1e-6:
        raise RuntimeError('Application inference does not match the selected interpreter')

    test_results = {
        'repeat_count':len(repeat),
        'max_probability_drift':drift,
        'synthetic_expected_probability':golden_score,
        'synthetic_actual_probability':float(repeat[0]),
        'validation_image_byte_path_consistency':True,
        'invalid_input_cases_checked':sorted(bad_cases)+['oversized_payload'],
        'status_example_synthetic':good['status'],
        'status_example_validation':prediction['status'],
        'limitations':'Python/Colab only. Android inference parity and low-end device benchmarks remain untested.'
    }
    (OUT/'inference_tests.json').write_text(json.dumps(test_results,indent=2)+'\n',encoding='utf-8')
    print('Repeatability + input checks passed:'); display(pd.DataFrame([test_results]))
else:
    print('Inference tests skipped. A valid candidate must pass model selection first.')

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Repeatability + input checks passed:


,repeat_count,max_probability_drift,synthetic_expected_probability,synthetic_actual_probability,validation_image_byte_path_consistency,invalid_input_cases_checked,status_example_synthetic,status_example_validation,limitations
0,8,0.0,0.417479,0.417479,True,"[empty, invalid_bytes, unsupported_format, ove...",Suspicious,Suspicious,Python/Colab only. Android inference parity an...


## 12. Prepare an **optional** FastAPI reference and the Flutter contract

The offline Android model is the product. FastAPI is a **separate optional future interface** for engineering tests; Flutter screening must never depend on it. Export server code separately from the **mobile-only ZIP** to keep download size and dependencies small.

In [12]:
if DEPLOY_READY:
    backend = OUT / 'backend_optional'
    backend.mkdir(exist_ok=True)
    shutil.copy2(module_path, backend/'skinx_inference.py')

    sample_api = """# Optional online diagnostic/testing route; NOT used by offline Flutter screening.
from pathlib import Path
from fastapi import FastAPI, UploadFile, File
from skinx_inference import ScreeningEngine

assets = Path(__file__).resolve().parent / 'assets'
engine = ScreeningEngine(assets/'skinx_screening.tflite',
                         assets/'skinx_model_config.json')
app = FastAPI(title='SkinX optional inference reference')

@app.post('/screen')
async def screen(file: UploadFile = File(...)):
    # Limit upload streaming in production reverse proxy too.
    max_bytes = int(engine.config['max_upload_bytes'])
    raw = await file.read(max_bytes+1)
    return engine.screen_bytes(raw)
"""
    (backend/'fastapi_example.py').write_text(sample_api,encoding='utf-8')
    (backend/'requirements_optional.txt').write_text(
        f'tensorflow=={tf.__version__}\nPillow\nnumpy\nfastapi\nuvicorn\npython-multipart\n',encoding='utf-8')
    (backend/'README.md').write_text("""# Optional Python/FastAPI reference

Screening in the Android app is **OFFLINE** and does not call this API.
For optional server testing only, copy `skinx_screening.tflite` and
`skinx_model_config.json` to `backend_optional/assets/` before launch.
`skinx_inference.py` is the single Python source of truth for input validation,
preprocessing and the decision label; it needs TensorFlow on the server.
For production, add authentication, rate limits, file size limits at the proxy,
secure storage controls, privacy review, and tests before exposing any endpoint.
No photos or patient data are automatically synced by the offline app.
""",encoding='utf-8')

    flutter_contract = """# Flutter Android offline inference contract

- Ship ONLY the two mobile assets in `SkinX_android_assets.zip`.
  Do not bundle TensorFlow Python or the optional backend.
- Handle capture/selection fully on the device; rotate according to EXIF BEFORE resize.
- Validate JPEG/PNG bytes and reject corrupt/unsupported files. The allowed formats
  and resource limits in the JSON are TECHNICAL guards, not medical quality scores.
- Pixel contract: 3-channel RGB, **224×224**, bilinear aspect-ratio resize,
  centered **black** padding. Float pixel range remains **0–255**.
- Read `input_dtype`, `input_quantization`, `output_dtype`, `output_quantization`
  from the saved config to quantize/dequantize. Do not divide by 255.
- Use `target_probability_threshold` exactly as saved. >= threshold means
  Suspicious, below means Less suspicious. Cannot infer is for invalid images
  or processing errors; there is NO validated uncertainty abstention cutoff.
- Compare Flutter's preprocessing on `parity_tests/synthetic_source.png` to
  `parity_tests/expected_input_rgb_float32.npy`, then its output to
  `parity_tests/expected_output.json` before accepting Android implementation.
- Test on physical low-memory, older Android devices for cold/hot time, RAM,
  model loading, camera orientation and battery cost. Colab timing is not that test.
- Store history locally and securely if consented; internet is optional, not a dependency.
- Low target probability is not a medical clearance. Even with a Less suspicious
  result, non-healing/changing lesions need a clinician.
- Do NOT show probabilities as a confirmed diagnosis or claim histopathology-level safety.
"""
    (OUT/'Flutter_offline_integration_contract.md').write_text(flutter_contract,encoding='utf-8')
    print('Optional reference + Flutter implementation contract exported:',backend)
else:
    print('Optional backend reference skipped: no accepted TFLite model.')

Optional reference + Flutter implementation contract exported: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/backend_optional


## 13. Save mobile-only assets and reproducibility files

Save model+config only in the Android bundle. Keep Python source, optional API example, repeatability report, validation comparisons, PWA descriptive check, parity vectors and SHA-256 manifest in the notebook's Drive run folder. No patient image is included in Android assets; there are no cloud requests in offline inference.

In [13]:
summary = {
    'run_tag':RUN_TAG,
    'source_lock_sha256':sha256(LOCK_JSON),
    'source_model_sha256':lock['model_sha256'],
    'train_manifest_sha256':sha256(TRAIN_CSV),
    'validation_manifest_sha256':sha256(VAL_CSV),
    'threshold_unchanged':THRESHOLD,
    'keras_validation_metrics':reference,
    'conversion_report':conversion,
    'deployment_ready':DEPLOY_READY,
    'tested_in_python_on_colab':bool(DEPLOY_READY),
    'android_tested':False,
    'uncertainty_abstention_validated':False,
    'medical_image_quality_classifier_validated':False,
    'no_pwa_or_test_tuning':True,
    'tensorflow_version':tf.__version__,
    'numpy_version':np.__version__,
    'pandas_version':pd.__version__,
    'note':'Model selection based on conventional validation only; no proven clinical safety or on-device benchmarks.'
}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
if DEPLOY_READY:
    mobile_files = ['skinx_screening.tflite', 'skinx_model_config.json']
    for item in mobile_files:
        if not (deploy_dir/item).exists():
            raise FileNotFoundError(f'Missing mobile asset {item}')
    model_hash = sha256(deploy_dir/'skinx_screening.tflite')
    if model_hash != config['tflite_sha256']:
        raise ValueError('Model hash mismatch before mobile export')
    mobile_archive = OUT / 'SkinX_android_assets.zip'
    from zipfile import ZipFile, ZIP_DEFLATED
    with ZipFile(mobile_archive, 'w', compression=ZIP_DEFLATED) as archive:
        for item in mobile_files:
            archive.write(deploy_dir/item,arcname=item)

    (OUT/'artifact_sha256.json').write_text(json.dumps({
        'skinx_screening.tflite':model_hash,
        'skinx_model_config.json':sha256(deploy_dir/'skinx_model_config.json'),
        'skinx_inference.py':sha256(module_path),
        'SkinX_android_assets.zip':sha256(mobile_archive),
        'synthetic_source.png':sha256(synthetic_file),
        'expected_input_rgb_float32.npy':sha256(parity_dir/'expected_input_rgb_float32.npy'),
    },indent=2)+'\n',encoding='utf-8')
    print('READY FOR FLUTTER IMPLEMENTATION (NOT clinically cleared).')
    print('Mobile-only bundle:',mobile_archive)
    print('Module for parity/optional FastAPI:',module_path)
    print('Android physical-device performance and Flutter parity are still REQUIRED.')
else:
    print('NOT READY: no conversion preserved original validation TP/TN.')
    print('See conversion_report.json and validation_comparison.csv; keep original locked model.')
print('Notebook 06 output:', OUT)

READY FOR FLUTTER IMPLEMENTATION (NOT clinically cleared).
Mobile-only bundle: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/SkinX_android_assets.zip
Module for parity/optional FastAPI: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2/skinx_inference.py
Android physical-device performance and Flutter parity are still REQUIRED.
Notebook 06 output: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_v2
